# Train First Model

This notebook trains the first machine learning model for binary ECG classification.

## 1. Import libraries

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load prepared dataset

We load the cleaned binary dataset created in `04_prepare_binary_dataset.ipynb`.

In [4]:
data_path = "../data/processed/12sl_norm_binary_clean.csv"

df = pd.read_csv(data_path)

## 3. Preview dataset

This table shows the first rows of the prepared dataset.

In [5]:
df.head()

,ecg_id,P_Area_I,P_PeakTime_I,Q_Area_I,Q_PeakTime_I,R_Area_I,R_PeakTime_I,S_Area_I,S_PeakTime_I,QRS_Balance_I,...,T+_Dur_aVF,T-_Dur_aVF,T+_Amp_aVF,T-_Amp_aVF,T_Morph_aVF,T_DurFull_aVF,P_Found_Global,HR__Global,P_Term_V1,target
0,1,0.264,64.0,0.000,0.0,0.737,30.0,0.000,0.0,629.0,...,208.0,0.0,0.151,0.000,1,208.0,1,64.0,0.000,0
1,21803,-0.047,6.0,0.015,8.0,0.794,36.0,1.246,76.0,205.0,...,0.0,127.0,0.000,-0.078,-1,127.0,1,75.0,0.000,1
2,21804,0.000,0.0,0.000,0.0,0.792,38.0,1.389,74.0,141.0,...,0.0,66.0,0.000,-0.078,-1,66.0,0,169.0,0.000,1
3,21805,0.196,74.0,0.000,0.0,0.947,56.0,0.000,0.0,439.0,...,0.0,0.0,0.000,0.000,-1,0.0,1,79.0,0.000,1
4,21806,0.287,102.0,0.000,0.0,1.021,36.0,0.000,0.0,937.0,...,104.0,66.0,39.000,-0.034,-2,170.0,1,92.0,6.612,1


In [6]:
df.shape

(21799, 778)

## 4. Prepare features and target

We separate input features from the target label.

- `X` contains ECG features used by the model.
- `y` contains the target label.

In [7]:
X = df.drop(columns=["ecg_id", "target"])
y = df["target"]

## 5. Check feature and target shapes

We check if the feature table and target column have matching sizes.

In [8]:
X.shape, y.shape

((21799, 776), (21799,))

## 6. Split dataset into training and test sets

We split the dataset into two parts:

- 80% of records are used for training
- 20% of records are used for testing

This split is used only in this notebook to test the first model!

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y #  keeps a similar normal/abnormal class distribution in both training and test sets
)

## 7. Check train and test sizes

We check how many records are in the training and test sets.

In [10]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((17439, 776), (4360, 776), (17439,), (4360,))

The dataset was split into training and test parts.  

## 8. Check class distribution after split

In [11]:
train_distribution = (y_train.value_counts(normalize=True) * 100).round(3)
test_distribution = (y_test.value_counts(normalize=True) * 100).round(3)

train_distribution, test_distribution

(target
 1    59.023
 0    40.977
 Name: proportion, dtype: float64,
 target
 1    59.014
 0    40.986
 Name: proportion, dtype: float64)

The class distribution is similar in both sets, because the split was stratified using the target column.

## 9. Prepare data for the model


Random Forest does not require feature scaling, so we use the original numerical feature values.


In [12]:
X_train_model = X_train
X_test_model = X_test


This is enough for Random Forest, because tree-based models split data by feature thresholds and do not need standardized input values.


## 10. Train Random Forest model

We train the first Random Forest model for binary ECG classification.


In [13]:
model = RandomForestClassifier(
    n_estimators=80,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_model, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",80
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fea

## 11. Make predictions

We use the trained model to predict labels for the test set.

In [14]:
y_pred = model.predict(X_test_model)


## 12. Check accuracy

Accuracy shows what part of test records was classified correctly.

In [15]:
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.8713302752293578

The first Random Forest model gives a simple reference result for binary ECG classification.

The exact score can change after changing parameters or the dataset split.


## 13. Classification report

We check precision, recall, and F1-score for both classes.

In [16]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.88      0.85      1787
           1       0.91      0.87      0.89      2573

    accuracy                           0.87      4360
   macro avg       0.87      0.87      0.87      4360
weighted avg       0.87      0.87      0.87      4360



## 14. Confusion matrix

We check how many records were classified correctly and incorrectly.

In [17]:
confusion_matrix(y_test, y_pred)

array([[1572,  215],
       [ 346, 2227]])

The confusion matrix shows correct and incorrect predictions for both classes.

## 15. Confusion matrix as a table

We display the confusion matrix in a more readable form.

In [18]:
cm = confusion_matrix(y_test, y_pred)

cm_table = pd.DataFrame(
    cm,
    index=["Actual normal", "Actual abnormal"],
    columns=["Predicted normal", "Predicted abnormal"]
)

cm_table

,Predicted normal,Predicted abnormal
Actual normal,1572,215
Actual abnormal,346,2227
